## Test the greedy layout design strategy

In [ ]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *

result_dir = f"result/layout/greedy/example_{get_timestamp()}"
os.makedirs(result_dir, exist_ok=True)

package = Package((50, 50), FabricateModel(20, 10), design_strategy="greedy", result_dir=result_dir)
# package = Package(design_strategy="greedy", result_dir=result_dir)

package.save_design_history()
package.save()
print("Package score:", package.estimate_score())

Initial score:  33.878021187989866
New action:  (ComputeModule_1_1_48_48, Strip, Y, 4)
New score:  80.83825999419155
New action:  (ComputeModule_31_1_34_48, Strip, X, 20)
New score:  104.43789658919748
Package score:  86.76993159278022


In [ ]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.element.package import *

result_dir = f"result/layout/greedy/example_20250617111701"
package = Package(pre_designed_package_path=os.path.join(result_dir, "package.pkl"))
print("Package score:", package.estimate_score())

## Test the rl-based layout design strategy

In [ ]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *
from qunoc.layout.env import QuNoCEnv
from qunoc.layout.agent import LayoutAgent

result_dir = f"result/layout/rl/example_{get_timestamp()}"
os.makedirs(result_dir, exist_ok=True)

env = QuNoCEnv((50, 50), FabricateModel(20, 10), result_dir=result_dir)
# env = QuNoCEnv(result_dir=result_dir)
agent = LayoutAgent(env, result_dir=result_dir)
agent.train(1000)
package = agent.design()

package.save_design_history()
package.save()
print("Package score:", package.estimate_score())

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Training agent...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 16.9     |
|    ep_rew_mean     | -1.4e+09 |
| time/              |          |
|    fps             | 8        |
|    iterations      | 1        |
|    time_elapsed    | 249      |
|    total_timesteps | 2048     |
---------------------------------
Designing layout...
Initial score:  39.61728499372216
New action:  [17 40  2  1  1]
New score:  74.54653705609819
New action:  [28 23  1  0  3]
New score:  65.51810600817757
New action:  [28 23  1  0  3]
New score:  71.03631243430449
New action:  [28 23  1  0  3]
New score:  70.1210073894442
New action:  [28 23  1  0  3]
New score:  80.50475805612783
New action:  [28 23  1  0  3]
New score:  70.70797684069667
New action:  [28 23  1  0  3]
New score:  68.75900348327541
New action:  [28 23  1  0  3]
New score:  67.74360363335184
New action:  [2

## Test the benchmark generation

In [1]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.element.benchmark import *

vqe_program = vqe(5)
print(vqe_program)
qft_program = qft(5)
print(qft_program)
qaoa_program = qaoa(5)
print(qaoa_program)
bv_program = bv(5)
print(bv_program)

vqe program: Control_0_1, Control_0_2, Control_0_3, Control_0_4, Control_1_2, Control_1_3, Control_1_4, Control_2_3, Control_2_4, Control_3_4
qft program: Control_1_0, Control_2_0, Control_3_0, Control_2_1, Control_4_0, Control_3_1, Control_4_1, Control_3_2, Control_4_2, Control_4_3
qaoa program: Control_2_4, Control_3_4, Control_0_2, Control_1_4, Control_1_2
bv program: Control_0_3, Control_0_4


## Test the routing algorithm

In [13]:
import os
import sys
sys.path.append('..')

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *
from qunoc.element.benchmark import *

algo = "vqe"
data_qubit_num = 20

# result_dir = f"result/routing/example_{get_timestamp()}"
result_dir = f"result/routing/example_{algo}_{data_qubit_num}"
os.makedirs(result_dir, exist_ok=True)

import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    filename=os.path.join(result_dir, "log.txt"),
    filemode='w',
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO,
    encoding='utf-8'
)

package = Package(package_size=(9, 9), fab_model=FabricateModel(9, 9), design_strategy="manual", result_dir=result_dir)
package.design_module_top_down(package.compute_modules[0], DesignTemplate.Grid, DesignOrientation.X, 3)

package.save_design_history()
package.save()
print("Package score:", package.estimate_score(), "\n")

program = get_program(algo, data_qubit_num)
gate_depend_fig = program.draw_depend()
gate_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_gate_depend.jpg"), bbox_inches='tight')
qiskit_circuit_fig = program.draw_circuit()
qiskit_circuit_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_qiskit_circuit.jpg"), bbox_inches='tight')

Package score: 92.71994962022112 



In [14]:
from qunoc.utils.config import config
from qunoc.routing.router import *

config.debug_mapping = True
config.debug_block = True
config.debug_action = True
router = Router(package=package, program=program, result_dir=result_dir)
router.map_qubits()
print(f"Routing program {program.algo_name}-{program.data_qubit_num}")

Routing program vqe-20


In [15]:
router.compile_blocks_and_actions()
router.save_block_history()
block_depend_fig = router.draw_block_depend()
block_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_block_depend.jpg"), bbox_inches='tight')
print("Block depth:", router.get_block_depth(), ", block count:", router.block_depend_graph.number_of_nodes())

Block depth: 18 , block count: 34


In [16]:
try:
    router.schedule_actions_to_circuit()
except AssertionError:
    router.save_action_history()
    raise
router.save_action_history()
action_depend_fig = router.draw_action_depend()
action_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_action_depend.jpg"), bbox_inches='tight')
print("Action depth:", router.get_action_depth(), ", action count:", router.action_depend_graph.number_of_nodes())
circuit = router.circuit
circuit.to_pickle(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_qunoc_circuit.pkl"))
print("QST fidelity:", router.estimate_average_gate_fide(), ", QuNoC circuit depth:", router.estimate_depth())

Action depth: 195 , action count: 1351
QST fidelity: 0.926136257746925 , QuNoC circuit depth: 0.28210000000000124


# Test the baseline MECH

In [1]:
import os
import sys
sys.path.append('..')

from baseline.MECH.MECHBenchmarks import *
import json
import os
%load_ext autoreload
%autoreload 2

directory_path = "exp_data"
if not os.path.exists(directory_path):
    os.makedirs(directory_path)

In [2]:
structure='hexagon'
chiplet_array_dim=(2,2)
chiplet_size=(3,3)

exp_result = MECH_experiments(structure=structure, chiplet_array_dim=chiplet_array_dim, chiplet_size=chiplet_size, benchmarks=['bv'], cross_link_sparsity=1)
print(exp_result)

22/45 gates are selected: [(0, 2), (0, 5), (0, 6), (0, 7), (0, 8), (1, 2), (1, 3), (1, 4), (1, 6), (1, 7)]...[(2, 6), (2, 7), (3, 4), (4, 7), (4, 9), (5, 6), (5, 8), (5, 9), (7, 9), (8, 9)]
----------------executing  bv
0/1
defaultdict(<class 'dict'>, {'bv': {'depth': 20, 'eff_gate_num': 28.4, 'on-chip': 10, 'cross-chip': 1, 'meas_num': 5, 'mech_circuit_depth': 4921.0, 'weight': (7.4, 2.2), 'periods': (13, 2), 'shuttle_num': 1, 'sparsity': 1}})


In [3]:
structure_abbr = '_'.join([s[:3] for s in structure.split('_')])
file_path = "./result/baseline/mech/{}{}{}{}{}.json".format(structure_abbr, chiplet_array_dim[0], chiplet_array_dim[1], chiplet_size[0], chiplet_size[1])
os.makedirs(os.path.dirname(file_path), exist_ok=True)

with open(file_path, 'w') as json_file:
    json.dump(exp_result, json_file)

print(f"The dictionary has been written to {file_path}")

The dictionary has been written to ./result/baseline/mech/hex2233.json
